# Location Analysis (new version)

This notebook processes and enriches location data from the Dehergne dataset to prepare it for geographic visualization and analysis.

**Workflow Stages:**

1.  **Data Extraction**:
    It gathers personal attributes related to locations (e.g., `nascimento`, `jesuita-entrada`, `morte`, `estadia`) into a unified DataFrame.

2.  **Date Inference**:
    Because many location attributes lack explicit dates, it infers them based on the chronological order of events in each person's biography (`ffill` method). It flags these inferred dates (`place.date_is_inferred`) for transparency.

3.  **Wikidata Enrichment**:
    It extracts Wikidata IDs from the source comments (e.g., `@wikidata:Q123`) to link historical place names to modern geodata.

4.  **Exports & Reports**:
    Generates several datasets in `../inferences/` for further use:
    *   `places_of_stay.xlsx`: Comprehensive list of stays with Wikidata IDs.
    *   `locations_names_places_dates_only.xlsx`: Simplified list for timeline visualization.
    *   `wikidata-references/locations_names_wikidata.csv`: Unique list of Wikidata IDs for the geocoding pipeline.
    *   `locations_names_no_wikidata.xlsx`: "To-do" list of places missing Wikidata links.
    *   `locations_totals.xlsx`: Frequency analysis of places (visit counts, date ranges).

5.  **Specific Analysis**:
    *   Filters missionaries by a specific place of entry (defaul: *Coimbra*).
    *   Exports individual markdown summaries associated with a specific place for review.



In [ ]:
import os
import re
import pandas as pd
from timelink.notebooks import TimelinkNotebook
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date

pd.set_option("display.max_rows", 150)

tlnb = TimelinkNotebook()
db = tlnb.db
tlnb.print_info()

## Attribute configuration
Attributes that store place names.

In [ ]:
ATTRIBUTES_WITH_PLACE_NAMES = [
    'nascimento',
    'jesuita-entrada',
    'partida',
    'chegada',
    'estadia',
    'estadia-x',
    'jesuita-votos-local',
    'jesuita-ordenacao-padre',
    'morte',
]


## Visualization parameters

Columns to show in dataframe visualization

In [ ]:

PREVIEW_COLS = [
    'place',
    'wikidata_id',
    'place.date_inferred.formatted',
    'place.date_inferred',
    'place.date_is_inferred',
    'name',
    'groupname',
    'place.type',
    'place.original',
    'place.line',
]

## Load place attributes and infer missing dates

We try to infer attributes with missing dates.
Sometimes the dates of some attribute are missing in the transcription, because in the original data they were not explicitly mentioned. 

But mostly the attributes are mentioned in cronological order so we can infer a relative date for attributes with missing dates (after date of the previous attribute).

This is the purpose of the following cell. It will create a new column called `place.date_inferred` which will recieve the an explicit date, if available or an inferred date. Another column called `place.date_is_inferred` with record if the date in `place.date_inferred` was inferred or not.

In [ ]:
places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name', 'groupname', 'the_line', 'the_order', 'extra_info'],
    the_type=[
        'jesuita-entrada',
        'partida',
        'chegada',
        'estadia',
        'estadia-x',
        'jesuita-votos-local',
        'jesuita-ordenacao-padre',
        'morte',
    ],
    column_name='place',
    db=db,
)

places_of_stay.place = places_of_stay.place.fillna('?')
places_of_stay = places_of_stay[places_of_stay.place != '?'].copy()

places_of_stay['id_col'] = places_of_stay.index
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])

# Infer missing dates using the previous known date for each person
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)
places_of_stay['place.date_inferred'] = places_of_stay.groupby('id_col')['place.date_inferred'].ffill()
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']

places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
places_of_stay['place.date_inferred.formatted'] = places_of_stay['place.date_inferred'].apply(format_timelink_date)

places_of_stay.set_index('id_col', inplace=True)

places_of_stay.info()

## Extract wikidata IDs from extra info

This will extract the wikidata id from the comments of attribute values

In [ ]:
import pandas as pd

from dehergne_util import extract_wikidata_id

places_of_stay['wikidata_id'] = places_of_stay['place.extra_info'].apply(extract_wikidata_id, if_missing=pd.NA)
places_of_stay.info()

In [ ]:
def dataframe_overview(df, df_name="df", print_report=True):
    info_df = pd.DataFrame({
        '#': range(len(df.columns)),
        'Column': df.columns,
        'Non-Null Count': df.notna().sum().values,
        'Dtype': df.dtypes.astype(str).values,
    })

    info_df['Non-Null Count'] = info_df['Non-Null Count'].astype(str) + ' non-null'

    dtype_summary = df.dtypes.astype(str).value_counts().sort_index()
    dtype_text = ", ".join(f"{k}({v})" for k, v in dtype_summary.items())
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    if print_report:
        if len(df.index) > 0:
            try:
                idx_min, idx_max = df.index.min(), df.index.max()
            except TypeError:
                idx_min, idx_max = df.index[0], df.index[-1]
            print(f"{df_name} -> Index: {len(df)} entries, {idx_min} to {idx_max}")
        else:
            print(f"{df_name} -> Index: 0 entries")

        print(f"Data columns (total {df.shape[1]} columns):")
        print(info_df.to_markdown(index=False))
        print(f"\ndtypes: {dtype_text}")
        print(f"memory usage: {mem_mb:.2f} MB")

    return info_df, dtype_summary, dtype_text, mem_mb


# Example with current dataframe
info_df, dtype_summary, dtype_text, mem_mb = dataframe_overview(places_of_stay, "places_of_stay")


### Preview sample

Show place names of people in the Dehergne repertoire, avoiding the people "referred to" in the individual biographies, and so keeping just the missionaries names. These are the people registred under the group name "n". 

In [ ]:
(
    places_of_stay[places_of_stay.groupname == 'n'][PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred', 'name'])
    .sample(50)
 )

## Save enriched datasets

In [ ]:
(
    places_of_stay[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .to_excel('../inferences/locations_places_of_stay.xlsx', sheet_name='wikidata', index=True)
 )

(
    places_of_stay[['place', 'place.date_inferred.formatted', 'name', 'place.type', 'place.original']]
    .sort_values(by=['place', 'place.date_inferred.formatted'])
    .to_excel('../inferences/locations_names_places_dates_only.xlsx', sheet_name='locations', index=True)
 )

(
    places_of_stay['wikidata_id']
    .dropna()
    .drop_duplicates()
    .to_csv('../inferences/wikidata-references/locations_wikidata_ids.csv', index=False)
 )

## Locations missing wikidata IDs

In [ ]:
places_without_wikidata = places_of_stay[
    places_of_stay['wikidata_id'].isna()
    & ~places_of_stay['place'].astype(str).str.startswith('[')
].copy()

(
    places_without_wikidata[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .sample(60)
 )

In [ ]:
(
    places_without_wikidata[PREVIEW_COLS]
    .sort_values(by=['place', 'place.date_inferred'])
    .to_excel('../inferences/wikidata-references/locations_names_no_wikidata.xlsx', sheet_name='no_wikidata', index=True)
 )


## Important

For this to work make sure that notebook `wikidata-linked-data.ipynb` has been run. It will read the wikidate exported above to `../inferences/wikidata-references/` and fetch the information of each location to locations_wikidata_info.xlsx which is used in the next cell to provide extra information about each location

## Frequency of places

### Merge places with wikidata information


In [ ]:
# get information about wikidata locations. See notebooks/wikidata-linked-data.ipynb for how this file was created
locations_wikidata_info = pd.read_excel(
    "../inferences/wikidata-references/locations_wikidata_info.xlsx"
)

# Normalize join keys
locations_wikidata_info["wikidata_id"] = locations_wikidata_info["wikidata_id"].astype("string").str.strip()
places_of_stay["wikidata_id"] = places_of_stay["wikidata_id"].astype("string").str.strip()

# Ensure many-to-one join (one row per wikidata_id on lookup table)
locations_wikidata_info = locations_wikidata_info.drop_duplicates(subset=["wikidata_id"], keep="first")

# Left join: keep all rows from places_of_stay and copy columns from locations_wikidata_info
places_of_stay = (
    places_of_stay.reset_index()
    .merge(
        locations_wikidata_info,
        how="left",
        left_on="wikidata_id",
        right_on="wikidata_id",
        suffixes=("", "_wikidata"),
    )
    .set_index("id_col")
)

places_of_stay.info()

### Show sample records



In [ ]:
cols = [ 'name', 'place', 'place.type','portuguese_label','chinese_label','label','country_label','wikidata_id','english_description', 'administrative_entity_id', 'administrative_entity_label_en']
places_of_stay[cols].sample(10)

In [ ]:
df_totals = (
    places_of_stay.reset_index()
    .groupby(['portuguese_label','chinese_label','english_label'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
    .sort_values('id_unique', ascending=False)
)

df_totals.head(100)

In [ ]:
df_totals.to_excel('../inferences/locations_totals.xlsx', sheet_name='totals', index=True)

### By country or region

In [ ]:
df_countries = (
    places_of_stay.reset_index()
    .groupby(['country_label'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
    .sort_values('id_unique', ascending=False)
)

df_countries

In [ ]:
df_countries.to_excel('../inferences/locations_countries.xlsx', sheet_name='countries', index=True)

## Analysis of name variations



In [ ]:
df_variations = (
    places_of_stay.reset_index()
    .groupby(['country_label','portuguese_label','chinese_label','english_label','wikidata_id','place'])
    .agg(id_unique=('id_col', 'nunique'), first_date=('place.date_inferred', 'min'), last_date=('place.date_inferred', 'max'))
)
df_variations.head(100)

In [ ]:
df_variations.reset_index().to_excel('../inferences/wikidata-references/locations_variations.xlsx', sheet_name='variations', index=True)

## Produce a Gazeeter 

Same data but alphabetical order of place, place in the first column, followed by wikidata_id, first_date, last_date, unique_id, and the labels

In [ ]:
df_variations.reset_index()[['place', 'wikidata_id', 'country_label', 'portuguese_label', 'chinese_label', 'english_label', 'id_unique','first_date','last_date']].sort_values(by=['place']).to_excel('../inferences/locations_geonames.xlsx', index=False)

### Place names variations for a specific country

In [ ]:
unique_countries = (
    df_variations.index.get_level_values("country_label")
    .dropna()
    .unique()
    .tolist()
)

sorted(unique_countries)

In [ ]:
country_of_interest = "People's Republic of China"
country_of_interest_df = df_variations[df_variations.index.get_level_values('country_label') == country_of_interest]
country_of_interest_df.head(100)

### Export country of interest places

In [ ]:
filename = f"../inferences/locations_{country_of_interest}_name_variations.xlsx"

country_of_interest_df.to_excel(filename, sheet_name='variations', index=True)

## Who was at a specific place

In [ ]:
place_of_interest = 'Coimbra'

In [ ]:
cols_place_detail = ['place', 'wikidata_id', 'name', 'place.type', 'place.date_inferred', 'place.obs']
(
    places_of_stay[places_of_stay.place == place_of_interest][cols_place_detail]
    .sort_values(by=['place.date_inferred'])
    .head(10)
)

In [ ]:
wikidata_id_of_interest='Q1207099'
cols_place_detail = ['place', 'wikidata_id', 'name', 'place.type', 'place.date_inferred', 'place.obs']
(
    places_of_stay[places_of_stay.wikidata_id == wikidata_id_of_interest][cols_place_detail]
    .sort_values(by=['place.date_inferred'])
    .head(100)
)

### Export markdown stubs for persons at that place

In [ ]:
markdown_directory = f"../inferences/{place_of_interest}/markdown"
os.makedirs(markdown_directory, exist_ok=True)

for person_id, name in places_of_stay[places_of_stay.place == place_of_interest][['name']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        person = tlnb.db.get_person(person_id, session=session)
        session.add(person)
        kleio_text = person.to_kleio()

    note = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{kleio_text}
```

---
Select title, right click on selected title, choose extract note
"""
    
    with open(f"{markdown_directory}/{person_id}.md", "w", encoding="utf-8") as handle:
        handle.write(note)

## Recover missing Wikidata IDs from variations

Try to find candidate Wikidata IDs for locations missing them, using the variations file that contains variations of names for known Wikidata IDs.

In [ ]:
import pandas as pd
import os
from difflib import SequenceMatcher

# Define file paths
no_wikidata_path = '../inferences/wikidata-references/locations_names_no_wikidata.xlsx'
variations_path = '../inferences/wikidata-references/locations_variations.xlsx'
output_path = '../inferences/wikidata-references/locations_names_no_wikidata_with_candidates.xlsx'

# Check if files exist
if not os.path.exists(no_wikidata_path):
    print(f"File not found: {no_wikidata_path}")
    # Try alternative path based on notebook logic
    no_wikidata_path = '../inferences/locations_names_no_wikidata.xlsx'
    print(f"Trying: {no_wikidata_path}")

if not os.path.exists(variations_path):
    print(f"File not found: {variations_path}")
    # Try alternative path
    variations_path = '../inferences/locations_variations.xlsx'
    print(f"Trying: {variations_path}")

try:
    df_missing = pd.read_excel(no_wikidata_path)
    df_variations = pd.read_excel(variations_path)

    print(f"Loaded {len(df_missing)} rows with missing Wikidata IDs.")
    print(f"Loaded {len(df_variations)} known location variations.")

    # 1. Basic Cleaning
    df_missing['place_clean'] = df_missing['place'].astype(str).str.strip()
    df_variations['place_clean'] = df_variations['place'].astype(str).str.strip()

    # 2. Prepare candidates
    # We only care about variations that actually have a Wikidata ID
    valid_variations = df_variations.dropna(subset=['wikidata_id'])[['place_clean', 'wikidata_id']].drop_duplicates()
    candidate_names = valid_variations['place_clean'].tolist()
    candidate_ids = valid_variations['wikidata_id'].tolist()
    
    # Create an exact lookup first for O(1) matching
    exact_lookup = dict(zip(candidate_names, candidate_ids))

    # 3. Fuzzy matching function
    def find_best_match(query, choices, ids, cutoff=0.6):
        """
        Find best string match using difflib.SequenceMatcher.
        Returns (id, match_name, score)
        """
        # Exact match optimization
        if query in exact_lookup:
            return exact_lookup[query], query, 1.0

        best_match = None
        best_id = None
        best_ratio = 0.0
        
        for choice, wid in zip(choices, ids):
            # SequenceMatcher gives a ratio between 0 and 1
            ratio = SequenceMatcher(None, query, choice).ratio()
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = choice
                best_id = wid
        
        if best_ratio >= cutoff:
            return best_id, best_match, best_ratio
        return None, None, 0.0

    print("Finding matches (this might take a while)...")
    
    # 4. Apply matching
    # Work on unique names to avoid re-calcuating the same name multiple times
    unique_missing = df_missing['place_clean'].unique()
    print(f"Processing {len(unique_missing)} unique missing place names...")
    
    results = []
    for i, place in enumerate(unique_missing):
        wid, match_name, score = find_best_match(place, candidate_names, candidate_ids, cutoff=0.65)
        results.append({
            'place_clean': place,
            'candidate_id': wid,
            'candidate_name': match_name,
            'match_score': score
        })
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(unique_missing)}...")
    
    df_results = pd.DataFrame(results)
    
    # 5. Merge results back
    df_final = df_missing.merge(df_results, on='place_clean', how='left')

    # 6. Report and Save
    found_matches = df_final[df_final['candidate_id'].notna()]
    print(f"Found {len(found_matches)} candidates out of {len(df_missing)} rows.")

    if len(found_matches) > 0:
        print("\nTop matches (by score):")
        cols_to_show = ['place', 'candidate_name', 'match_score', 'candidate_id']
        print(found_matches[cols_to_show].drop_duplicates(subset=['place']).sort_values('match_score', ascending=False).head(15))

    df_final.drop(columns=['place_clean'], inplace=True, errors='ignore')
    df_final.to_excel(output_path, index=False)
    print(f"\nSaved {len(df_final)} rows to {output_path}")

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
df_final.info()


We generate a document with the biographies of all the peopl with references to non located places